In [5]:
import pandas as pd
import numpy as np

In [6]:
import mlxtend.frequent_patterns as frequent

In [7]:
aisles = pd.read_csv('aisles.csv')
departments = pd.read_csv('departments.csv')
orders = pd.read_csv('orders.csv')
opp = pd.read_csv('order_products__prior.csv')
opt = pd.read_csv('order_products__train.csv')
products = pd.read_csv('products.csv')

In [8]:
aisles.info()
departments.info()
orders.info()
opp.info()
opt.info()
products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 134 entries, 0 to 133
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   aisle_id  134 non-null    int64 
 1   aisle     134 non-null    object
dtypes: int64(1), object(1)
memory usage: 2.2+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   department_id  21 non-null     int64 
 1   department     21 non-null     object
dtypes: int64(1), object(1)
memory usage: 468.0+ bytes
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3421083 entries, 0 to 3421082
Data columns (total 7 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   order_id                int64  
 1   user_id                 int64  
 2   eval_set                object 
 3   order_number            int64  
 4   order_dow               in

# Task 1: choose a subset of data (20,000 users)

## Pick a manageable subset (20,000 users)

In [9]:

N_USERS = 20000
np.random.seed(42)

# Get unique users and sample
sample_users = np.random.choice(orders['user_id'].unique(), size=N_USERS, replace=False)

# Keep only orders from these users
orders_sub = orders[orders['user_id'].isin(sample_users)].copy()

orders_sub.shape, orders_sub['user_id'].nunique()


((335251, 7), 20000)

## Keep only PRIOR orders + join products to those orders

In [10]:
# Use only 'prior' orders for market basket analysis
orders_prior = orders_sub[orders_sub['eval_set'] == 'prior'][['order_id', 'user_id']].copy()

# Filter order_products__prior to only those order_ids
opp_sub = opp[opp['order_id'].isin(orders_prior['order_id'])].copy()

orders_prior.shape, opp_sub.shape


((315251, 2), (3191742, 4))

## Basic cleaning: nulls + obvious validity checks

In [11]:
# 1) Remove null values in key columns
opp_sub = opp_sub.dropna(subset=['order_id', 'product_id'])

# 2) Ensure correct dtypes (sometimes helpful after merges/reads)
opp_sub['order_id'] = opp_sub['order_id'].astype(np.int64)
opp_sub['product_id'] = opp_sub['product_id'].astype(np.int64)

# 3) Remove duplicates: same product appearing multiple times in same order (shouldn't happen, but safe)
opp_sub = opp_sub.drop_duplicates(subset=['order_id', 'product_id'])

opp_sub.shape


(3191742, 4)

## Remove “bad invoices”: orders with only 1 item

In [12]:
# Count items per order
items_per_order = opp_sub.groupby('order_id')['product_id'].nunique()

# Keep only orders with at least 2 distinct products
valid_orders = items_per_order[items_per_order >= 2].index

opp_clean = opp_sub[opp_sub['order_id'].isin(valid_orders)].copy()

opp_sub.shape, opp_clean.shape


((3191742, 4), (3176865, 4))

In [13]:
print("Users in subset:", orders_sub['user_id'].nunique())
print("Prior orders in subset:", orders_prior['order_id'].nunique())

print("\nBefore removing 1-item orders:")
print("Orders:", opp_sub['order_id'].nunique(), "Rows:", len(opp_sub))

print("\nAfter removing 1-item orders:")
print("Orders:", opp_clean['order_id'].nunique(), "Rows:", len(opp_clean))

print("\nAverage basket size (after cleaning):",
      opp_clean.groupby('order_id')['product_id'].nunique().mean())


Users in subset: 20000
Prior orders in subset: 315251

Before removing 1-item orders:
Orders: 315251 Rows: 3191742

After removing 1-item orders:
Orders: 300374 Rows: 3176865

Average basket size (after cleaning): 10.57636479855114


In [14]:
print("Note:")
print("- Instacart dataset does NOT contain product price or invoice amount, so 'remove zero/negative price' is not applicable.")
print("- It also does not contain an explicit 'returns/negative quantity' column; we ensure validity via null checks, deduplication, and basket-size filtering.")


Note:
- Instacart dataset does NOT contain product price or invoice amount, so 'remove zero/negative price' is not applicable.
- It also does not contain an explicit 'returns/negative quantity' column; we ensure validity via null checks, deduplication, and basket-size filtering.


# Task 2.1 & 2.2: Group by order_id and collect products

## Create order → product list (transaction format)

In [16]:

basket = (
    opp_clean
    .groupby('order_id')['product_id']
    .apply(list)
)

basket.head


<bound method NDFrame.head of order_id
28         [35108, 40593, 17461, 22825, 25256, 47626, 372...
32         [12384, 15991, 13176, 20995, 18362, 35887, 496...
48         [14129, 22298, 17600, 6141, 42169, 12144, 1140...
54         [10833, 24852, 34004, 35430, 11121, 36086, 32971]
58                 [23375, 2825, 22825, 13176, 26604, 42265]
                                 ...                        
3420988    [3957, 34668, 21903, 24184, 15290, 31506, 2620...
3420999                                [32478, 43692, 37752]
3421066             [10017, 46667, 41950, 18479, 23233, 210]
3421080    [27845, 4932, 18811, 41950, 31717, 12935, 2512...
3421083    [7854, 45309, 21162, 18176, 35211, 39678, 1135...
Name: product_id, Length: 300374, dtype: object>

In [17]:
basket.head()

order_id
28    [35108, 40593, 17461, 22825, 25256, 47626, 372...
32    [12384, 15991, 13176, 20995, 18362, 35887, 496...
48    [14129, 22298, 17600, 6141, 42169, 12144, 1140...
54    [10833, 24852, 34004, 35430, 11121, 36086, 32971]
58            [23375, 2825, 22825, 13176, 26604, 42265]
Name: product_id, dtype: object

## One-hot encode the baskets

In [18]:

from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
te_array = te.fit(basket).transform(basket)

basket_encoded = pd.DataFrame(
    te_array,
    columns=te.columns_,
    index=basket.index
)

basket_encoded.shape


(300374, 41189)

## Quick sanity checks

In [19]:
# Sanity checks
print("Number of transactions:", basket_encoded.shape[0])
print("Number of unique products:", basket_encoded.shape[1])

# Check that values are binary
basket_encoded.iloc[:5, :5]


Number of transactions: 300374
Number of unique products: 41189


,1,2,3,4,5
order_id,,,,,
28,False,False,False,False,False
32,False,False,False,False,False
48,False,False,False,False,False
54,False,False,False,False,False
58,False,False,False,False,False


# Task 3 — Running Apriori Algorithm

### With 41,189 products, running Apriori directly at 0.01 is:
#### extremely slow
#### may crash or freeze kernel
#### This is not a mistake — it’s a reality of Apriori.
#### Standard, acceptable solution (academically correct)
#### Before Apriori:
#### Remove very rare products
#### Keep only products that appear in at least 1% of transactions
#### This is considered part of preprocessing and is fully acceptable.

### Remove extremely rare products

In [20]:
# Product frequency (support of 1-itemsets)
item_support = basket_encoded.mean()

# Keep items that appear in at least 1% of transactions
frequent_items = item_support[item_support >= 0.01].index

basket_reduced = basket_encoded[frequent_items]

basket_encoded.shape, basket_reduced.shape


((300374, 41189), (300374, 108))

### Apriori with min_support = 0.01

In [21]:
from mlxtend.frequent_patterns import apriori

frequent_itemsets_001 = apriori(
    basket_reduced,
    min_support=0.01,
    use_colnames=True
)

frequent_itemsets_001['length'] = frequent_itemsets_001['itemsets'].apply(len)

frequent_itemsets_001.shape


(125, 3)

### Apriori with min_support = 0.05

In [22]:
frequent_itemsets_005 = apriori(
    basket_reduced,
    min_support=0.05,
    use_colnames=True
)

frequent_itemsets_005['length'] = frequent_itemsets_005['itemsets'].apply(len)

frequent_itemsets_005.shape


(6, 3)

### Compare results (core of Task 3)

In [23]:
print("min_support = 0.01")
print("Number of itemsets:", len(frequent_itemsets_001))
print("Max itemset size:", frequent_itemsets_001['length'].max())

print("\nmin_support = 0.05")
print("Number of itemsets:", len(frequent_itemsets_005))
print("Max itemset size:", frequent_itemsets_005['length'].max())


min_support = 0.01
Number of itemsets: 125
Max itemset size: 2

min_support = 0.05
Number of itemsets: 6
Max itemset size: 1


In [24]:
basket_encoded.shape, basket_reduced.shape


((300374, 41189), (300374, 108))

# Task 4 — Association Rules

### Generate association rules

In [25]:
from mlxtend.frequent_patterns import association_rules

rules = association_rules(
    frequent_itemsets_001,
    metric="lift",
    min_threshold=1
)

rules.shape


(32, 14)

### Keep essential columns (clean table)

In [28]:
rules_clean = rules[[
    'antecedents',
    'consequents',
    'support',
    'confidence',
    'lift'
]].copy()

rules_clean.sort_values(by='lift', ascending=False).head()


,antecedents,consequents,support,confidence,lift
14,(21137),(27966),0.011206,0.128444,2.765479
15,(27966),(21137),0.011206,0.241273,2.765479
6,(13176),(47209),0.021240,0.171110,2.394118
7,(47209),(13176),0.021240,0.297187,2.394118
4,(13176),(27966),0.013653,0.109988,2.368105


In [29]:
rules_clean.sort_values(by='lift', ascending=False).head

<bound method NDFrame.head of    antecedents consequents   support  confidence      lift
14     (21137)     (27966)  0.011206    0.128444  2.765479
15     (27966)     (21137)  0.011206    0.241273  2.765479
6      (13176)     (47209)  0.021240    0.171110  2.394118
7      (47209)     (13176)  0.021240    0.297187  2.394118
4      (13176)     (27966)  0.013653    0.109988  2.368105
5      (27966)     (13176)  0.013653    0.293957  2.368105
16     (21137)     (47209)  0.013490    0.154621  2.163413
17     (47209)     (21137)  0.013490    0.188746  2.163413
30     (49683)     (24852)  0.010517    0.325368  2.160065
31     (24852)     (49683)  0.010517    0.069820  2.160065
21     (21903)     (47209)  0.011146    0.144317  2.019235
20     (47209)     (21903)  0.011146    0.155953  2.019235
29     (47766)     (24852)  0.016869    0.291157  1.932941
28     (24852)     (47766)  0.016869    0.111990  1.932941
1      (21137)     (13176)  0.020581    0.235900  1.900399
0      (13176)     (21137)

### Find top 3 rules by lift

In [33]:
top3_lift = rules_clean.sort_values(by='lift', ascending=False).head(3)
top3_lift


,antecedents,consequents,support,confidence,lift
14,(21137),(27966),0.011206,0.128444,2.765479
15,(27966),(21137),0.011206,0.241273,2.765479
6,(13176),(47209),0.021240,0.171110,2.394118
